In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

plt.style.use("ggplot")

In [ ]:
DATA_PATH = Path("../data/processed/feature_engineered_data.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

In [ ]:
consumption_column = [
    col for col in df.columns
    if (
        "energy" in col.lower()
        or "consumption" in col.lower()
        or "kwh" in col.lower()
    )
][0]

print("Consumption Column:", consumption_column)

In [ ]:
feature_columns = [
    consumption_column,
    "hour",
    "month",
    "is_peak_hour"
]

feature_columns = [
    col for col in feature_columns
    if col in df.columns
]

X = df[feature_columns]

X.head()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

In [ ]:
model = IsolationForest(
    n_estimators=150,
    contamination=0.02,
    random_state=42
)

model.fit(X_scaled)

print("Model training completed.")

In [ ]:
df["anomaly_flag"] = model.predict(X_scaled)

df["anomaly_label"] = df["anomaly_flag"].map({
    1: "normal",
    -1: "anomaly"
})

df["anomaly_score"] = model.decision_function(X_scaled)

df.head()

In [ ]:
df["anomaly_label"].value_counts()

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    x="anomaly_label",
    y=consumption_column,
    data=df
)

plt.title("Normal vs Anomalous Consumption")

plt.show()

In [ ]:
plt.figure(figsize=(16, 6))

normal_data = df[
    df["anomaly_label"] == "normal"
]

anomaly_data = df[
    df["anomaly_label"] == "anomaly"
]

plt.scatter(
    normal_data.index,
    normal_data[consumption_column],
    alpha=0.3,
    label="Normal"
)

plt.scatter(
    anomaly_data.index,
    anomaly_data[consumption_column],
    alpha=0.9,
    label="Anomaly"
)

plt.title("Detected Energy Consumption Anomalies")

plt.xlabel("Record Index")
plt.ylabel("Consumption")

plt.legend()

plt.show()

In [ ]:
if "is_peak_hour" in df.columns:

    anomaly_peak = (
        df.groupby(
            ["is_peak_hour", "anomaly_label"]
        )
        .size()
        .unstack()
    )

    anomaly_peak.plot(
        kind="bar",
        figsize=(10, 5)
    )

    plt.title("Peak Hour Anomaly Analysis")

    plt.show()

In [ ]:
if "season" in df.columns:

    seasonal_anomalies = (
        df[df["anomaly_label"] == "anomaly"]
        .groupby("season")
        .size()
    )

    seasonal_anomalies.plot(
        kind="bar",
        figsize=(10, 5)
    )

    plt.title("Seasonal Anomaly Distribution")

    plt.show()

In [ ]:
top_anomalies = (
    df[df["anomaly_label"] == "anomaly"]
    .sort_values("anomaly_score")
    .head(10)
)

top_anomalies[
    [
        consumption_column,
        "hour",
        "month",
        "anomaly_score"
    ]
]

In [ ]:
print("ANOMALY DETECTION INSIGHTS")
print("=" * 50)

total_records = len(df)

anomaly_records = (
    df["anomaly_label"] == "anomaly"
).sum()

anomaly_percentage = (
    anomaly_records / total_records
) * 100

print(f"Total Records: {total_records:,}")
print(f"Anomalies Detected: {anomaly_records:,}")
print(f"Anomaly Percentage: {anomaly_percentage:.2f}%")

peak_anomalies = (
    df[
        (df["anomaly_label"] == "anomaly")
        &
        (df["is_peak_hour"] == 1)
    ]
)

print(
    f"Peak Hour Anomalies: {len(peak_anomalies):,}"
)

print(
    "\\nPotential recommendation:"
)

print(
    "Peak-hour households may benefit from "
    "demand-response optimization strategies."
)